# 04 — Feature Engineering

## Leakage-safe annual region forecasting features

This notebook creates the final feature table for later forecasting notebooks. It uses only
the frozen Phase 1 annual and seasonal panels and never modifies them.

The forecasting unit is **administrative region × year**. Lagged, rolling, growth, and
historical regional features use only observations earlier than the target year. This prevents
future information from leaking into a forecast.


## 1. Imports and reproducible configuration


In [ ]:
from pathlib import Path
from datetime import datetime
from copy import copy
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 2. Locate project and load validated panels


In [ ]:
def locate_project_root(start: Path) -> Path:
    '''Locate the repository without relying on an absolute machine path.'''
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter from the project root or notebooks directory."
    )


PROJECT_ROOT = locate_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

for directory in (RESULTS_DIR, FIGURES_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ANNUAL_PATH = PROCESSED_DIR / "saudi_household_admin_region_annual_panel.csv"
SEASONAL_PATH = PROCESSED_DIR / "saudi_household_admin_region_seasonal_panel.csv"

if not ANNUAL_PATH.exists() or not SEASONAL_PATH.exists():
    raise FileNotFoundError(
        "Validated Phase 1 panels are required. Run the frozen Phase 1 notebooks first."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Annual input: {ANNUAL_PATH.relative_to(PROJECT_ROOT)}")
print(f"Seasonal input: {SEASONAL_PATH.relative_to(PROJECT_ROOT)}")

FEATURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
annual = pd.read_csv(ANNUAL_PATH).sort_values(["region", "year"]).reset_index(drop=True)
seasonal = pd.read_csv(SEASONAL_PATH).sort_values(
    ["region", "year", "season"]
).reset_index(drop=True)

print(f"Annual input: {annual.shape}")
print(f"Seasonal input: {seasonal.shape}")


## 3. Input quality gate

The notebook stops if required values are missing, keys are duplicated, values are non-finite,
or consumption/cost is not strictly positive. Provenance-only missing fields do not affect the
analytical quality gate.


In [ ]:
annual_key = ["year", "region"]
seasonal_key = ["year", "region", "season"]
annual_required = [
    "year", "region", "winter_consumption_kwh", "rest_of_year_consumption_kwh",
    "winter_cost_sar", "rest_of_year_cost_sar", "annual_consumption_kwh",
    "annual_cost_sar", "average_cost_sar_per_kwh", "winter_consumption_share_pct",
]
seasonal_required = ["year", "region", "season", "consumption_kwh", "cost_sar"]

assert annual[annual_required].isna().sum().sum() == 0
assert seasonal[seasonal_required].isna().sum().sum() == 0
assert annual.duplicated(annual_key).sum() == 0
assert seasonal.duplicated(seasonal_key).sum() == 0
assert np.isfinite(annual[annual_required[2:]].to_numpy()).all()
assert np.isfinite(seasonal[["consumption_kwh", "cost_sar"]].to_numpy()).all()
assert (annual[annual_required[2:8]] > 0).all().all()
assert (seasonal[["consumption_kwh", "cost_sar"]] > 0).all().all()

print("Input quality gate passed.")


## 4. Feature definitions and scientific design

- Contemporaneous annual and seasonal component values describe the target observation and
  support exploratory or cost forecasting tasks.
- Lag features provide the most recent historical state.
- Two-year rolling statistics summarize recent level and volatility and are shifted by one year.
- Growth features represent momentum using only prior observations.
- Historical regional averages are expanding means shifted by one year.
- Ratios are scale-normalized measures of seasonal structure and cost intensity.
- `region_id` is a stable deterministic label; `region` is retained for interpretable encoding
  inside later model pipelines.

No arbitrary polynomial terms are created.


In [ ]:
feature_df = annual[[
    "year", "region",
    "annual_consumption_kwh", "annual_cost_sar",
    "winter_consumption_kwh", "rest_of_year_consumption_kwh",
    "winter_cost_sar", "rest_of_year_cost_sar",
    "average_cost_sar_per_kwh", "winter_consumption_share_pct",
    "selected_source", "source_file",
]].copy()

feature_df["time_index"] = feature_df["year"] - feature_df["year"].min()
region_categories = sorted(feature_df["region"].unique())
region_map = {region: index for index, region in enumerate(region_categories)}
feature_df["region_id"] = feature_df["region"].map(region_map).astype(int)

feature_df["rest_to_winter_consumption_ratio"] = (
    feature_df["rest_of_year_consumption_kwh"] / feature_df["winter_consumption_kwh"]
)
feature_df["winter_to_rest_consumption_ratio"] = (
    feature_df["winter_consumption_kwh"] / feature_df["rest_of_year_consumption_kwh"]
)
feature_df["winter_cost_share_pct"] = (
    100 * feature_df["winter_cost_sar"] / feature_df["annual_cost_sar"]
)

grouped = feature_df.groupby("region", sort=False)
for lag in (1, 2):
    feature_df[f"consumption_lag_{lag}"] = grouped["annual_consumption_kwh"].shift(lag)
    feature_df[f"cost_lag_{lag}"] = grouped["annual_cost_sar"].shift(lag)

feature_df["consumption_rolling_mean_2"] = grouped["annual_consumption_kwh"].transform(
    lambda values: values.shift(1).rolling(2, min_periods=2).mean()
)
feature_df["consumption_rolling_std_2"] = grouped["annual_consumption_kwh"].transform(
    lambda values: values.shift(1).rolling(2, min_periods=2).std(ddof=1)
)
feature_df["cost_rolling_mean_2"] = grouped["annual_cost_sar"].transform(
    lambda values: values.shift(1).rolling(2, min_periods=2).mean()
)

feature_df["consumption_growth_lag_1_pct"] = grouped["annual_consumption_kwh"].transform(
    lambda values: values.shift(1).pct_change() * 100
)
feature_df["cost_growth_lag_1_pct"] = grouped["annual_cost_sar"].transform(
    lambda values: values.shift(1).pct_change() * 100
)
feature_df["winter_consumption_growth_lag_1_pct"] = grouped[
    "winter_consumption_kwh"
].transform(lambda values: values.shift(1).pct_change() * 100)
feature_df["rest_consumption_growth_lag_1_pct"] = grouped[
    "rest_of_year_consumption_kwh"
].transform(lambda values: values.shift(1).pct_change() * 100)

feature_df["region_historical_mean_consumption"] = grouped[
    "annual_consumption_kwh"
].transform(lambda values: values.shift(1).expanding(min_periods=1).mean())
feature_df["region_historical_mean_cost"] = grouped[
    "annual_cost_sar"
].transform(lambda values: values.shift(1).expanding(min_periods=1).mean())

feature_df["source_lineage"] = (
    feature_df["selected_source"].astype(str) + " | " + feature_df["source_file"].astype(str)
)


## 5. Establish the model-ready history window

Two-year lags and two-observation rolling statistics require two complete earlier years.
Therefore, the 2017–2018 warm-up rows are excluded from the final table. They remain intact in
the frozen Phase 1 panels. This produces a complete, model-ready 2019–2022 table without
imputation or invented history.


In [ ]:
engineered_columns = [
    "consumption_lag_1", "consumption_lag_2",
    "cost_lag_1", "cost_lag_2",
    "consumption_rolling_mean_2", "consumption_rolling_std_2",
    "cost_rolling_mean_2",
    "consumption_growth_lag_1_pct", "cost_growth_lag_1_pct",
    "winter_consumption_growth_lag_1_pct",
    "rest_consumption_growth_lag_1_pct",
    "region_historical_mean_consumption", "region_historical_mean_cost",
]

warmup_mask = feature_df[engineered_columns].isna().any(axis=1)
warmup_rows = feature_df.loc[warmup_mask, ["year", "region"]].copy()
model_features = feature_df.loc[~warmup_mask].copy().reset_index(drop=True)

print(f"Full region-year rows: {len(feature_df)}")
print(f"Warm-up rows excluded: {len(warmup_rows)}")
print(f"Final model-ready rows: {len(model_features)}")
print(f"Final years: {model_features['year'].min()}–{model_features['year'].max()}")


## 6. Feature quality control


In [ ]:
numeric_features = model_features.select_dtypes(include=[np.number]).columns.tolist()
key_duplicates = int(model_features.duplicated(["year", "region"]).sum())
missing_cells = int(model_features.isna().sum().sum())
infinite_cells = int(
    np.isinf(model_features[numeric_features].to_numpy(dtype=float)).sum()
)
constant_columns = [
    column for column in model_features.columns
    if model_features[column].nunique(dropna=False) <= 1
]
impossible_counts = {
    "nonpositive_consumption_target": int((model_features["annual_consumption_kwh"] <= 0).sum()),
    "nonpositive_cost": int((model_features["annual_cost_sar"] <= 0).sum()),
    "negative_time_index": int((model_features["time_index"] < 0).sum()),
    "winter_share_outside_0_100": int(
        (~model_features["winter_consumption_share_pct"].between(0, 100)).sum()
    ),
    "winter_cost_share_outside_0_100": int(
        (~model_features["winter_cost_share_pct"].between(0, 100)).sum()
    ),
}

qc = pd.DataFrame([
    ("rows", len(model_features)),
    ("columns", model_features.shape[1]),
    ("duplicate_keys", key_duplicates),
    ("missing_cells", missing_cells),
    ("infinite_cells", infinite_cells),
    ("constant_columns", len(constant_columns)),
    *[(name, value) for name, value in impossible_counts.items()],
], columns=["check", "value"])
display(qc)
print("Constant columns:", constant_columns)

if key_duplicates or missing_cells or infinite_cells or any(impossible_counts.values()):
    raise ValueError("Feature quality control failed.")


## 7. Feature distribution verification


In [ ]:
distribution_columns = [
    "consumption_lag_1", "consumption_lag_2",
    "consumption_rolling_mean_2", "consumption_rolling_std_2",
    "consumption_growth_lag_1_pct", "average_cost_sar_per_kwh",
]
display(model_features[distribution_columns].describe().T)

for column in distribution_columns:
    skewness = model_features[column].skew()
    if abs(skewness) > 2:
        print(f"Review note: {column} is strongly skewed (skewness={skewness:.2f}).")


## 8. Document every feature


In [ ]:
feature_definitions = [
    ("year", "Identity", "Published calendar year", "Temporal location", "Chronological splitting and trend"),
    ("region", "Identity", "Canonical administrative region label", "Preserves geography", "Categorical regional effect"),
    ("time_index", "Time", "year − 2017", "Monotonic elapsed-time measure", "Linear/nonlinear trend input"),
    ("region_id", "Regional", "Alphabetical zero-based region code", "Stable region identifier", "Pipeline-compatible grouping identifier"),
    ("annual_consumption_kwh", "Target/current", "winter_consumption_kwh + rest_of_year_consumption_kwh", "Annual household demand", "Primary future forecasting target"),
    ("annual_cost_sar", "Current", "winter_cost_sar + rest_of_year_cost_sar", "Annual household electricity expenditure", "Cost-level target or explanatory variable"),
    ("winter_consumption_kwh", "Seasonal", "Published winter consumption", "Cold-season demand component", "Seasonal demand structure"),
    ("rest_of_year_consumption_kwh", "Seasonal", "Published rest-of-year consumption", "Non-winter demand component", "Seasonal demand structure"),
    ("winter_cost_sar", "Seasonal", "Published winter cost", "Cold-season expenditure", "Seasonal cost structure"),
    ("rest_of_year_cost_sar", "Seasonal", "Published rest-of-year cost", "Non-winter expenditure", "Seasonal cost structure"),
    ("average_cost_sar_per_kwh", "Consumption/cost", "annual_cost_sar ÷ annual_consumption_kwh", "Observed cost intensity", "Tariff/cost relationship"),
    ("winter_consumption_share_pct", "Seasonal", "100 × winter_consumption_kwh ÷ annual_consumption_kwh", "Winter share of annual demand", "Scale-normalized seasonality"),
    ("rest_to_winter_consumption_ratio", "Interaction", "rest_of_year_consumption_kwh ÷ winter_consumption_kwh", "Relative seasonal demand", "Scale-normalized seasonality"),
    ("winter_to_rest_consumption_ratio", "Interaction", "winter_consumption_kwh ÷ rest_of_year_consumption_kwh", "Inverse seasonal balance", "Interpretable seasonal contrast"),
    ("winter_cost_share_pct", "Interaction", "100 × winter_cost_sar ÷ annual_cost_sar", "Winter share of annual cost", "Scale-normalized cost seasonality"),
    ("consumption_lag_1", "Lag", "annual_consumption_kwh at t−1", "Most recent annual demand", "Persistence and autoregression"),
    ("consumption_lag_2", "Lag", "annual_consumption_kwh at t−2", "Second prior annual demand", "Multi-year memory"),
    ("cost_lag_1", "Lag", "annual_cost_sar at t−1", "Most recent annual cost", "Cost persistence"),
    ("cost_lag_2", "Lag", "annual_cost_sar at t−2", "Second prior annual cost", "Multi-year cost memory"),
    ("consumption_rolling_mean_2", "Rolling", "mean(consumption at t−1, t−2)", "Recent demand level", "Smoothed historical baseline"),
    ("consumption_rolling_std_2", "Rolling", "sample SD(consumption at t−1, t−2)", "Recent demand variability", "Historical volatility"),
    ("cost_rolling_mean_2", "Rolling", "mean(cost at t−1, t−2)", "Recent cost level", "Smoothed historical cost baseline"),
    ("consumption_growth_lag_1_pct", "Growth", "100 × (consumption t−1 ÷ consumption t−2 − 1)", "Prior observed demand growth", "Momentum without target leakage"),
    ("cost_growth_lag_1_pct", "Growth", "100 × (cost t−1 ÷ cost t−2 − 1)", "Prior observed cost growth", "Cost momentum"),
    ("winter_consumption_growth_lag_1_pct", "Growth", "100 × (winter t−1 ÷ winter t−2 − 1)", "Prior winter demand growth", "Season-specific momentum"),
    ("rest_consumption_growth_lag_1_pct", "Growth", "100 × (rest t−1 ÷ rest t−2 − 1)", "Prior non-winter demand growth", "Season-specific momentum"),
    ("region_historical_mean_consumption", "Regional", "expanding mean of consumption through t−1", "Past regional demand scale", "Leakage-safe region-level baseline"),
    ("region_historical_mean_cost", "Regional", "expanding mean of cost through t−1", "Past regional cost scale", "Leakage-safe region-level baseline"),
    ("selected_source", "Lineage", "Phase 1 selected source", "Preserves source decision", "Audit only; exclude from numeric model matrix"),
    ("source_file", "Lineage", "Phase 1 source filename", "Links record to input file", "Audit only; exclude from model matrix"),
    ("source_lineage", "Lineage", "selected_source + source_file", "Human-readable provenance", "Audit only; exclude from model matrix"),
]
feature_dictionary = pd.DataFrame(
    feature_definitions,
    columns=[
        "feature_name", "category", "formula", "description",
        "forecasting_rationale",
    ],
)
feature_dictionary.insert(
    4,
    "scientific_motivation",
    feature_dictionary["description"].map(
        lambda text: f"Represents {text[:1].lower() + text[1:]} in an interpretable form."
    ),
)

undocumented = sorted(set(model_features.columns) - set(feature_dictionary["feature_name"]))
if undocumented:
    raise ValueError(f"Undocumented output columns: {undocumented}")
display(feature_dictionary)


## 9. Save the feature dataset, dictionary, and engineering report


In [ ]:
FEATURE_PATH = FEATURES_DIR / "household_feature_dataset.csv"
DICTIONARY_PATH = RESULTS_DIR / "feature_dictionary.xlsx"
REPORT_PATH = RESULTS_DIR / "feature_engineering_report.md"

model_features.to_csv(FEATURE_PATH, index=False, float_format="%.15g")

with pd.ExcelWriter(DICTIONARY_PATH, engine="openpyxl") as writer:
    feature_dictionary.to_excel(writer, sheet_name="Feature Dictionary", index=False)
    qc.to_excel(writer, sheet_name="Quality Control", index=False)
    pd.DataFrame(
        [(region, code) for region, code in region_map.items()],
        columns=["region", "region_id"],
    ).to_excel(writer, sheet_name="Region Mapping", index=False)
    warmup_rows.to_excel(writer, sheet_name="Excluded Warm-up Rows", index=False)

    for worksheet in writer.book.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.sheet_view.showGridLines = False
        for cell in worksheet[1]:
            cell.font = copy(cell.font)
            cell.font = __import__("openpyxl").styles.Font(
                name=cell.font.name, size=cell.font.size, bold=True, color="FFFFFF"
            )
            cell.fill = __import__("openpyxl").styles.PatternFill(
                "solid", fgColor="1F4E78"
            )
        for column_cells in worksheet.columns:
            values = [str(cell.value) if cell.value is not None else "" for cell in column_cells]
            width = min(max(max(map(len, values)) + 2, 12), 48)
            worksheet.column_dimensions[column_cells[0].column_letter].width = width
            for cell in column_cells:
                cell.alignment = __import__("openpyxl").styles.Alignment(
                    vertical="top", wrap_text=True
                )
        if worksheet.title == "Feature Dictionary":
            for letter, width in {
                "A": 34, "B": 18, "C": 52, "D": 36, "E": 58, "F": 42
            }.items():
                worksheet.column_dimensions[letter].width = width
            for row in range(2, worksheet.max_row + 1):
                worksheet.row_dimensions[row].height = 38

report_lines = [
    "# Feature Engineering Report",
    "",
    f"Generated: {datetime.now().astimezone().isoformat(timespec='seconds')}",
    "",
    "## Scope and frozen inputs",
    "",
    "- Input: `data/processed/saudi_household_admin_region_annual_panel.csv`",
    "- Input: `data/processed/saudi_household_admin_region_seasonal_panel.csv`",
    "- The Phase 1 files were read without modification.",
    "",
    "## Forecasting unit",
    "",
    "One row represents one administrative region in one year.",
    "",
    "## Leakage controls",
    "",
    "- All lag, rolling, growth, and historical regional features use values through t−1.",
    "- Rolling windows are shifted before calculation.",
    "- No random split, target-derived future value, or arbitrary polynomial interaction is used.",
    "",
    "## History requirement",
    "",
    f"- Full input rows: {len(feature_df)}",
    f"- Excluded warm-up rows: {len(warmup_rows)} (2017–2018)",
    f"- Final model-ready rows: {len(model_features)}",
    f"- Final coverage: {model_features['year'].min()}–{model_features['year'].max()}, "
    f"{model_features['region'].nunique()} regions",
    "- Warm-up rows were excluded rather than imputed because two-year history does not exist.",
    "",
    "## Quality control",
    "",
]
report_lines.extend([f"- **{row.check}**: {row.value}" for row in qc.itertuples()])
report_lines.extend([
    "",
    "## Constant columns",
    "",
    f"- {', '.join(constant_columns) if constant_columns else 'None'}",
    "",
    "## Modeling cautions",
    "",
    "- The final table is small; later evaluation must be chronological and uncertainty-aware.",
    "- Current-year consumption and cost components must not be used to predict the same-year "
    "annual consumption target. Later model notebooks must define predictors according to the "
    "forecast origin.",
    "- `region_id` is an identifier, not an ordinal measure. Later models should use appropriate "
    "categorical encoding or region-aware modeling.",
    "- Lineage text fields are retained for auditability and should be excluded from numeric models.",
])
REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

print(f"Saved: {FEATURE_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved: {DICTIONARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved: {REPORT_PATH.relative_to(PROJECT_ROOT)}")


## Conclusion

The final feature table is complete, finite, uniquely keyed, documented, and reproducible from
the frozen Phase 1 panels. Its historical predictors are leakage-safe. Forecasting models are
deliberately not implemented in this notebook.
